# N173 · Hard序列、字符串与设计综合

**目标：** 独立推导状态或候选支配并比较两种解法。

**先修：** N029, N056, N107, N170, N171。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最短覆盖；单调队列；前缀；区间合并；状态设计；适用条件。

## 从问题到算法

Hard 题的难点常是多个小前提同时成立，而非更长代码。最小覆盖窗口记录**缺少的字符实例数**，不是缺少的字符种类；目标 `AABC` 中两个 A 都必须被覆盖。

带负数最短子数组不能照搬普通窗口，改用前缀支配队列。合并石头则先检查可达性：每次把 k 堆变一堆，总堆数减少 k−1，因此 `(n−1)` 必须被 k−1 整除。

区间 DP 保存每段能压缩到的最少合法堆数的最小成本；分割点步长 k−1，只有区间能最终压成一堆时才加总重量。输出值正确之外，还必须解释每次“加区间和”的物理含义。

## 最小实现

**本章接口：** `min_window_audited(s, t)`、`shortest_subarray_audited(nums, k)`、`merge_stones_audited(stones, k)`

In [1]:
from collections import Counter,deque

def min_window_audited(s,t):
    if not t:return ''
    required=Counter(t);missing=len(t);left=0;best=None
    for right,char in enumerate(s):
        if required[char]>0:missing-=1
        required[char]-=1
        while missing==0:
            if best is None or right-left+1<best[1]-best[0]:best=(left,right+1)
            old=s[left];required[old]+=1;left+=1
            if required[old]>0:missing+=1
    return '' if best is None else s[best[0]:best[1]]

def shortest_subarray_audited(nums,k):
    if k<=0:raise ValueError('Positive threshold required')
    prefixes=[0]
    for value in nums:prefixes.append(prefixes[-1]+value)
    queue=deque();best=len(nums)+1
    for j,value in enumerate(prefixes):
        while queue and value-prefixes[queue[0]]>=k:best=min(best,j-queue.popleft())
        while queue and prefixes[queue[-1]]>=value:queue.pop()
        queue.append(j)
    return -1 if best>len(nums) else best

def merge_stones_audited(stones,k):
    if k<2:raise ValueError('At least two adjacent piles per merge')
    if any(value<0 for value in stones):raise ValueError('Nonnegative pile weights required')
    n=len(stones)
    if n<=1:return 0
    if (n-1)%(k-1):return -1
    prefix=[0]
    for value in stones:prefix.append(prefix[-1]+value)
    dp=[[0]*n for _ in range(n)]
    for length in range(2,n+1):
        for left in range(n-length+1):
            right=left+length-1
            dp[left][right]=min(dp[left][middle]+dp[middle+1][right]
                for middle in range(left,right,k-1))
            if (length-1)%(k-1)==0:dp[left][right]+=prefix[right+1]-prefix[left]
    return dp[0][-1]

## 正确性、前提与复杂度

窗口缺额为零等价于每个目标字符重数都满足。单调队列删除的是被较新、更小前缀支配的旧状态。石头区间长度 L 可压缩到 `(L−1) mod (k−1)+1` 堆；分割令左段可压一堆，穷举其最后分界覆盖所有合法结构。只有余数为零时还能把 k 堆合成一堆，成本恰是区间总和。

**代价：** 前两个算法时间 O(n+|t|)、O(n)，空间分别 O(字符种类)、O(n)。合并石头 O(n³/(k−1)) 时间（上界 O(n³)）、O(n²) 空间。独立穷举与规模冒烟分开。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='ADOBECODEBANC';t='ABC'
show_table(['mechanism','input','result'],[
 ['multiplicity window',(s,t),min_window_audited(s,t)],
 ['prefix dominance',([2,-1,2],3),shortest_subarray_audited([2,-1,2],3)],
 ['interval merge',([3,2,4,1],2),merge_stones_audited([3,2,4,1],2)],
 ['unreachable remainder',([3,2,4,1],3),merge_stones_audited([3,2,4,1],3)]])

mechanism,input,result
multiplicity window,"('ADOBECODEBANC', 'ABC')",BANC
prefix dominance,"([2, -1, 2], 3)",3
interval merge,"([3, 2, 4, 1], 2)",20
unreachable remainder,"([3, 2, 4, 1], 3)",-1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from functools import cache
from itertools import product
from random import Random
def brute_window(s,t):
    if not t:return ''
    need=Counter(t);best=''
    for i in range(len(s)):
        for j in range(i+1,len(s)+1):
            found=Counter(s[i:j])
            if all(found[x]>=count for x,count in need.items()):
                if not best or j-i<len(best):best=s[i:j]
                break
    return best
for n in range(7):
    for chars in product('AB',repeat=n):
        s=''.join(chars)
        for t in ['','A','AA','AB','AAB']:
            assert min_window_audited(s,t)==brute_window(s,t)
assert min_window_audited('ADOBECODEBANC','ABC')=='BANC'
assert shortest_subarray_audited([-2,-1],1)==-1
rng=Random(173)
for _ in range(250):
    nums=[rng.randrange(-5,8) for _ in range(rng.randrange(9))];k=rng.randrange(1,15)
    expected=min((j-i for i in range(len(nums)) for j in range(i+1,len(nums)+1) if sum(nums[i:j])>=k),default=-1)
    assert shortest_subarray_audited(nums,k)==expected
@cache
def brute_merge(state,k):
    if len(state)<=1:return 0
    answer=float('inf')
    for i in range(len(state)-k+1):
        merged=sum(state[i:i+k]);cost=brute_merge(state[:i]+(merged,)+state[i+k:],k)
        answer=min(answer,merged+cost)
    return answer
for n in range(8):
    for k in [2,3,4]:
        for _ in range(8):
            stones=tuple(rng.randrange(1,6) for _ in range(n))
            expected=brute_merge(stones,k)
            expected=-1 if expected==float('inf') else expected
            assert merge_stones_audited(stones,k)==expected
assert merge_stones_audited([3,2,4,1],2)==20
assert merge_stones_audited([3,2,4,1],3)==-1
# Scale smoke cases never call the exponential or cubic-string references.
assert min_window_audited('x'*20000+'AAB','AAB')=='AAB'
assert shortest_subarray_audited([1]*20000,100)==100
assert merge_stones_audited([1]*30,2)>0
print('N173: 所有本章断言通过')

N173: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 完成一题不看标签的变体。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 为每题提供一个导致常见错误的最小输入。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 76. Minimum Window Substring（comparison）
- 862. Shortest Subarray with Sum at Least K（comparison）：[官方入口](https://leetcode.com/problems/shortest-subarray-with-sum-at-least-k/)
- 1000. Minimum Cost to Merge Stones（comparison）：[官方入口](https://leetcode.com/problems/minimum-cost-to-merge-stones/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。